In [ ]:
from helper import DataLoader
import models.SimpleGCN.MultiRelCompGCN as MultiRelCompGCN
import torch
from sklearn.metrics import root_mean_squared_error

In [12]:
DATAPATH = "../data"
RATING_DATA = DATAPATH + "/train_ratings.csv"
TBR_DATA = DATAPATH + "/train_tbr.csv"

In [13]:
data_manager = DataLoader.DataLoader(data_dir=DATAPATH)
train_df, valid_df = data_manager.load_and_split()

In [14]:
print(train_df)

         sid  pid  rating
0          0    1       5
1          0   13       4
2          0   22       5
3          0   26       4
4          0   62       4
...      ...  ...     ...
896738   957  337       3
896739  2218  232       5
896740  1146  255       4
896741  5758  519       4
896742  1062  437       5

[896743 rows x 3 columns]


In [15]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [16]:
edge_t0_index, edge_t0_weights, edge_t1_index, edge_t1_weights = data_manager.get_graph_tensors(device)

In [3]:
EPOCHS = 200

# model = SimpleGCN.SimpleGCN(
#     num_users=data_manager.num_users,
#     num_items=data_manager.num_items,
#     embedding_dim=64,
#     num_layers=1
# ).to(device)



# --- 1. INITIALIZATION ---
# Use the MultiRelGCN class you defined earlier
model = MultiRelCompGCN.MultiRelCompGCN(
    num_users=data_manager.num_users,
    num_items=data_manager.num_items,
    embedding_dim=64,
    num_layers=2,
    dropout=0.1
).to(device)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-2,
)
criterion = torch.nn.MSELoss()
# --- 2. UPDATED PREDICTION WRAPPER ---
def predict_ratings(sids, pids):
    model.eval()
    with torch.no_grad():
        s_tensor = torch.tensor(sids, dtype=torch.long).to(device)
        p_tensor = torch.tensor(pids, dtype=torch.long).to(device)
        # Pass all 4 graph tensors
        out = model(s_tensor, p_tensor, 
                    edge_t0_index, edge_t0_weights, 
                    edge_t1_index, edge_t1_weights)
        return out.cpu().numpy() * 5.0

# --- 3. UPDATED TRAINING LOOP ---
for epoch in range(EPOCHS):
    model.train()
    optimizer.zero_grad()
    
    # Forward pass using both relation types
    # We predict ratings for the users/items present in our t0 (ratings) edges
    preds = model(
        edge_t0_index[0], edge_t0_index[1], 
        edge_t0_index, edge_t0_weights,
        edge_t1_index, edge_t1_weights
    )
    
    # Loss is still calculated against the actual ratings (weights_t0)
    loss = criterion(preds, edge_t0_weights)
    
    loss.backward()
    optimizer.step()
    
    if epoch % 10 == 0:
        val_preds = predict_ratings(valid_df["sid"].values, valid_df["pid"].values)
        val_rmse = np.sqrt(mean_squared_error(valid_df["rating"].values, val_preds))
        print(f"Epoch {epoch} | Loss: {loss.item():.4f} | Val RMSE: {val_rmse:.4f}")


NameError: name 'MultiRelCompGCN' is not defined